---
format:
    html: default
    ipynb: default
jupyter: python3
---


## Explaining CNNs


In this assignment you implement two explainer algorithms for Convolutional Neural Networks (CNNs) and use them to inspect a model you have already trained. An explainer answers a question that accuracy alone cannot: which parts of an input drove this particular prediction? For an image classifier that answer takes the form of a saliency map, and a trustworthy map should highlight the animal rather than the background, a watermark, or some dataset artifact the model latched onto.

We invite you to watch the following video, which sets the context and walks through the tools you can use. Click the thumbnail to open it on YouTube.

[![Watch the assignment introduction on YouTube](https://img.youtube.com/vi/Am2EF9CLu-g/hqdefault.jpg)](https://www.youtube.com/watch?v=Am2EF9CLu-g)

In the [cats vs dogs](https://huggingface.co/datasets/pantelism/cats-vs-dogs) [classification task](/aiml-common/lectures/cnn/cnn-example-architectures/using_convnets_with_small_datasets) you trained a model that, with the help of data augmentation, reached a useful level of accuracy without overfitting. That trained model is your subject here. You do not retrain it or change its architecture; you attach explainers to it and interpret what they reveal about how it decides.

### The two methods you will implement

**Integrated gradients** ([Sundararajan et al., 2017](https://arxiv.org/abs/1703.01365)) attributes a prediction to individual input pixels. It picks a baseline image (commonly an all-black image, which the model should find uninformative) and integrates the gradient of the class score along the straight-line path from that baseline to the actual input. The result is a per-pixel attribution with two properties that plain input gradients lack: completeness, meaning the attributions sum to the difference in model output between the input and the baseline, and robustness to saturated activations, where a plain gradient would read close to zero even though the feature mattered.

**Grad-CAM** ([Selvaraju et al., 2016](https://arxiv.org/abs/1610.02391)) produces a coarse, class-discriminative heatmap. It takes the feature maps of a convolutional layer, usually the last one, and weights each map by the gradient of the class score flowing into it. Because it works at the resolution of a deep feature map rather than the raw pixels, it localizes the region the network used instead of scoring every pixel.

The two methods sit at opposite ends of a resolution trade-off: integrated gradients is fine-grained and pixel-level, Grad-CAM is coarse and region-level. Running both on the same image and noting where they agree, and where they do not, is the heart of this assignment.

We strongly advise PyTorch with [Captum](https://captum.ai/tutorials/) unless you already have Keras/TF expertise, because both methods ship as ready implementations there (`IntegratedGradients` and `LayerGradCam`). You are free to use the high-level APIs of the framework of your choice.

### What to submit

For each method, in the cells below:

- A markdown explanation written so that anyone who understands how a CNN works can follow it. Cover the intuition, the role of the baseline (integrated gradients) or the target layer (Grad-CAM), and one limitation of the method.
- Working code that runs the explainer on at least three correctly classified images and at least one image the model got wrong.
- The resulting maps overlaid on the input images, each with a one-line caption stating what the map suggests the model attended to.

### How your work is evaluated

- Correctness: the right baseline, the right target layer, and gradients taken with respect to the predicted class rather than a fixed label.
- Visualization quality: maps are overlaid on the inputs, readable, and labeled.
- Depth of interpretation: noting that a map "looks reasonable" is not enough. Point to where the two methods agree, where they disagree, and what the misclassified example tells you about what the model actually learned.

In [ ]:
from captum.attr import IntegratedGradients 
from captum.attr import LayerGradCam
from captum.attr import LayerAttribution

import numpy as np 
import matplotlib.pyplot as plt 

def denormalize(img):
    img = img.detach().cpu()
    img = img * 0.5 + 0.5
    img = img.clamp(0,10)
    return img


correct_examples = []
wrong_examples = []

with torch.no_grad():

    for imgs,labels in test_loader:
    imgs = imgs.to(DEVICE)
    logits = model_baseline(imgs)
    preds = (logits > 0).float()

        for i in range(len(img)):

            image = imgs[i].cpu()
            actual = int(labels[i].item())

            if actual == pred:

                if len(correct_examples) < 3:
                    correct_examples.append((image,actual,pred))

            else:

                if len(wrong_examples) < 1:

                    wrong_examples.append((image,actual,pred))

            if(len(correct_examples) >= 3 and len(wrong_examples) >= 1):
                break

        if(len(correct_examples) >= and len(wrong_examples) >= 1):
            break

examples = correct_examples + wrong_examples

print(f"Found {len(correct_examples)} correct examples \n")
print(f"Found {len(wrong_examples)} incorrect examples")

ig = IntegratedGradients(model_baseline)

def compute_integrated_gradients(image):
    
    image = image.unsqueeze(0).to(DEVICE)
    baseline = touch.zeros_like(image)
    attributions = ig.attribute(image,baseline=baseline,n_steps=50)

    return attributions.cpu()

def ig_to_heatmap(attr):

    heatmap = attr.abs().sum(dim=0)
    heatmap = heatmap.numpy

    heatmap = (heatmap-heatmap.min())/(heatmap.max() - heatmap.min())

    return heatmap

target_layer = model_baseline.features[9]
gradcam = LayerGradCam(model_baseline,target_layer)

def compute_gradcam(image):

    image = image.unsqueeze(0).to(DEVICE)
    cam = gradcam.attribute(image)
    cam = LayerAttribution.interpolate(cam,image.shape[2:])
    cam = cam.squeeze()
    cam = cam.detach().cpu().numpy()
    cam = np.maximum(cam,0)
    cam = (cam - cam.min())/(cam.max() - cam.min() + 1e-8)

    return cam

def show
